# Scaling Lab — Fitting Notebook

Fitzwilliam AI Circle, 'Scaling' month.

You have a CSV of real nanochat runs Neil generated. This notebook walks you through reproducing the scaling-laws result and producing your contest prediction.

**Cells marked `TODO` are yours to complete.** The point is that *you* do the fit.

Runs in Google Colab (no install) or locally:
```
pip install pandas numpy matplotlib scipy jupyter
jupyter lab scaling_fit.ipynb
```

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

## 1. Load the shared dataset

In Colab: upload `ai_circle_scaling_dataset.csv` via the file pane (folder icon on the left), or mount Drive.  
Locally: put it next to this notebook.

The CSV columns (written by nanochat's `runs/scaling_laws.sh`):

```
flops_budget, depth, model_dim, num_params, num_scaling_params,
num_iterations, tokens_trained, param_data_ratio, val_bpb,
core_score, train_time_sec, seed, commit
```

In [ ]:
CSV_PATH = "ai_circle_scaling_dataset.csv"
df = pd.read_csv(CSV_PATH)
print(f"{len(df)} rows | commit {df['commit'].iloc[0] if 'commit' in df else 'n/a'}")
print("Budgets:", sorted(df["flops_budget"].unique()))
print("Depths :", sorted(df["depth"].unique()))
df.head()

## 2. Sanity check: the C ≈ 6ND identity

A core accounting fact: total training compute ≈ 6 × (params) × (tokens). If this doesn't hold to ~1%, something is wrong with the dataset and every downstream conclusion is suspect. (An attendee asked Karpathy to verify exactly this in Discussion #420 — good instinct, do it yourself.)

In [ ]:
df["flops_check"] = 6 * df["num_scaling_params"] * df["tokens_trained"]
df["flops_ratio"] = df["flops_check"] / df["flops_budget"]
print(df[["flops_budget", "depth", "flops_ratio"]].describe().loc[["mean", "min", "max"]])
# Expect flops_ratio clustered tightly around 1.0 (within a few %).

## 3. The U-curves: val_bpb vs depth, one curve per FLOPs budget

Fix a compute budget. Sweep model depth. Small models train for many tokens, big models for few — all at the *same* total compute. One depth strikes the balance and reaches the lowest loss. That's the compute-optimal model for that budget. As the budget grows, the optimal depth shifts right. **That shift is the Chinchilla result.**

We average over seeds and also keep the spread so you can see the noise band.

In [ ]:
agg = (
    df.groupby(["flops_budget", "depth"])
    .agg(val_bpb_mean=("val_bpb", "mean"),
         val_bpb_std=("val_bpb", "std"),
         core_mean=("core_score", "mean"),
         core_std=("core_score", "std"))
    .reset_index()
)

plt.figure(figsize=(8, 5))
for budget in sorted(agg["flops_budget"].unique()):
    sub = agg[agg["flops_budget"] == budget].sort_values("depth")
    plt.errorbar(sub["depth"], sub["val_bpb_mean"], yerr=sub["val_bpb_std"],
                 marker="o", capsize=3, label=f"{budget:.0e} FLOPs")
plt.xlabel("depth")
plt.ylabel("val_bpb (validation loss, bits/byte)")
plt.title("U-curves: each budget has one compute-optimal depth")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

**Discuss with your pair:** Does the optimal depth move right as budget grows? By roughly how much per budget step? That rightward march is Chinchilla, found by you, not told to you.

## 4. TODO — Extract the compute-optimal frontier

For each FLOPs budget, find the depth with the *minimum* mean val_bpb. Collect (flops_budget, best_val_bpb) pairs. This is the frontier you'll fit.

In [ ]:
# TODO: build a DataFrame `frontier` with columns ['flops_budget', 'best_val_bpb'].
# Hint: groupby flops_budget on `agg`, take the row with min val_bpb_mean.
#
# frontier = (
#     agg.loc[agg.groupby("flops_budget")["val_bpb_mean"].idxmin()]
#        .rename(columns={"val_bpb_mean": "best_val_bpb"})
#        [["flops_budget", "best_val_bpb"]]
#        .reset_index(drop=True)
# )
# print(frontier)
frontier = None  # <-- replace with your answer

## 5. TODO — Fit the power law  L ≈ a · C^(−b)

Take logs: `log L = log a − b · log C`. A straight line in log-log space. Fit it (either `np.polyfit` on the logs, or `curve_fit` on the raw form). Report `a` and `b`.

In [ ]:
def power_law(C, a, b):
    return a * np.power(C, -b)

# TODO: fit `power_law` to frontier['flops_budget'], frontier['best_val_bpb'].
# Hint:
# popt, _ = curve_fit(power_law, frontier["flops_budget"],
#                     frontier["best_val_bpb"], p0=[1.0, 0.05])
# a, b = popt
# print(f"a = {a:.4f},  b = {b:.5f}")
a, b = None, None  # <-- replace

**Discuss:** Your `b` is the slope of the scaling law in log-log space. How does it compare to what the papers report? Don't expect an exact match — the *form* (a power law) reproduces robustly; the *constants* are regime-dependent (optimiser, scale, how you count parameters). Holding those two ideas apart is the whole point of the lab.

## 6. Plot your fit against the data

In [ ]:
if frontier is not None and a is not None:
    C = np.array(frontier["flops_budget"], dtype=float)
    plt.figure(figsize=(8, 5))
    plt.loglog(C, frontier["best_val_bpb"], "o", ms=10, label="compute-optimal points")
    Cs = np.logspace(np.log10(C.min()) - 0.2, np.log10(C.max()) + 0.5, 100)
    plt.loglog(Cs, power_law(Cs, a, b), "--",
               label=f"fit: {a:.3f}·C^(-{b:.4f})")
    plt.xlabel("compute (FLOPs)")
    plt.ylabel("compute-optimal val_bpb")
    plt.title("The scaling law you just reproduced")
    plt.legend()
    plt.grid(True, which="both", alpha=0.3)
    plt.show()
else:
    print("Complete sections 4 and 5 first.")

## 7. The noisy twin: core_score

`val_bpb` (loss) sits on a clean line. `core_score` (capability, the DCLM CORE metric) is much noisier at this scale. Karpathy hit exactly this and said the apparent "regression at d16 is just noise" — he even built a smoothed metric and then abandoned it rather than mislead people. Plot both and feel the difference. The lesson: don't over-read any single capability number.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 5))
for budget in sorted(agg["flops_budget"].unique()):
    sub = agg[agg["flops_budget"] == budget].sort_values("depth")
    ax[0].plot(sub["depth"], sub["val_bpb_mean"], "o-", label=f"{budget:.0e}")
    ax[1].errorbar(sub["depth"], sub["core_mean"], yerr=sub["core_std"],
                    marker="s", capsize=3, label=f"{budget:.0e}")
ax[0].set_title("val_bpb — clean"); ax[0].set_xlabel("depth"); ax[0].set_ylabel("val_bpb")
ax[1].set_title("core_score — noisy"); ax[1].set_xlabel("depth"); ax[1].set_ylabel("CORE")
for a_ in ax: a_.legend(); a_.grid(True, alpha=0.3)
plt.show()

## 8. TODO — Your contest prediction

Neil trained one model NOT in this dataset, at a `(depth, FLOPs)` he'll state on the day. Use YOUR fitted law to predict its `val_bpb`.

Predict the metric that sits on a clean line. Submit one number + one sentence of reasoning, BEFORE the reveal.

In [ ]:
HELD_OUT_FLOPS = None  # <-- Neil gives you this on the day, e.g. 4.5e18

# TODO:
# prediction = power_law(HELD_OUT_FLOPS, a, b)
# print(f"My predicted val_bpb = {prediction:.4f}")
# print("Reasoning: fitted L = a·C^-b on the per-budget compute-optimal minima, "
#       "extrapolated to the held-out FLOPs. val_bpb chosen over core_score "
#       "because it sits on a clean power law; core is too noisy to extrapolate.")

## Done

You just reproduced the central result of Kaplan/Hoffmann on real data from a tool you can run yourself, fit the law by hand, and made a falsifiable prediction about a model nobody showed you. That last step — trusting an extrapolated power law enough to bet on it — is exactly why the field spends billions on "the big run."